In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
## preguntas que responde este hecho (Registro_Novedades)
###### 9. Cuales son las novedades que mas se presentan durante la prestacion del servicio


# database connections 

In [ ]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [ ]:
tabla_novedad = pd.read_sql_table('mensajeria_novedadesservicio', co_sa)
servicios     = pd.read_sql_table('mensajeria_servicio', co_sa)
dim_fecha     = pd.read_sql_table('dim_fecha', etl_conn)
tabla_novedad.info()

# Transformations

In [ ]:
hecho = tabla_novedad[['id', 'fecha_novedad', 'tipo_novedad_id', 'servicio_id', 'mensajero_id']].copy()

servicios_min = servicios[['id', 'cliente_id']].rename(columns={'id': 'servicio_id'})
hecho = hecho.merge(servicios_min, on='servicio_id', how='left')

hecho['num_novedades'] = 1

hecho['fecha_novedad'] = pd.to_datetime(hecho['fecha_novedad'], utc=True).dt.tz_localize(None)
hecho['id_hora'] = hecho['fecha_novedad'].dt.hour * 60 + hecho['fecha_novedad'].dt.minute

dim_fecha_min = dim_fecha[['key_dim_fecha', 'fecha_completa']].copy()
dim_fecha_min['fecha_completa'] = pd.to_datetime(dim_fecha_min['fecha_completa']).dt.date
hecho['fecha'] = hecho['fecha_novedad'].dt.date
hecho = hecho.merge(dim_fecha_min, left_on='fecha', right_on='fecha_completa', how='left')

hecho.info()

In [ ]:
hecho_novedad = hecho[[
    'key_dim_fecha', 'id_hora', 'cliente_id',
    'tipo_novedad_id', 'mensajero_id', 'servicio_id', 'num_novedades'
]]

hecho_novedad.info()
hecho_novedad.head()

# load

In [ ]:
hecho_novedad.to_sql('hecho_novedad', etl_conn, if_exists='replace', index_label='key_hecho_novedad')